# Detecção de Faces com a InsightFace

Neste notebook, vamos explorar a detecção facial com a biblioteca [`insightface`](https://github.com/deepinsight/insightface), especializada em rostos: detecção, pontos de referência (*landmarks*), reconhecimento e estimativa de idade/gênero.

Diferente da família YOLO, que trata "pessoa" ou "rosto" como só mais uma classe entre dezenas de objetos genéricos, a `insightface` foi desenhada e treinada especificamente para rostos, o que a torna muito mais precisa em cenários difíceis (rostos pequenos, ângulos extremos, multidões) e permite ir além da caixa delimitadora: ela também localiza pontos de referência que servem de base para tarefas como o alinhamento de faces, o pré-processamento padrão de sistemas de reconhecimento facial.

Vamos cobrir três casos de uso, cada um com uma imagem diferente:
1. **Detecção de múltiplas faces**, na foto de um time de futebol inteiro.
2. **Seleção da pessoa principal**, em uma selfie com várias pessoas.
3. **Alinhamento de faces**, normalizando poses diferentes de um mesmo rosto.

In [ ]:
from IPython import get_ipython
if 'google.colab' in str(get_ipython()):
    print("Preparando ambiente Google Colab")
    !pip install opencv-python==5.0.0.93
    !pip install opencv-contrib-python==5.0.0.93
    !pip install insightface
    !pip install onnxruntime
    !git clone https://github.com/pvoloshyn/curso-visao-computacional.git
    %cd curso-visao-computacional
else:
    pass

## Carregando bibliotecas

Além do OpenCV e do matplotlib de sempre, vamos usar a `insightface`, que por baixo dos panos roda seus modelos em ONNX através do `onnxruntime`.
* `FaceAnalysis`: a classe principal, que carrega um pacote de modelos e expõe o método `get()`, devolvendo a lista de rostos encontrados em uma imagem.
* `face_align`: módulo com a função `norm_crop()`, usada para alinhar um rosto a partir dos seus pontos de referência.

In [ ]:
from insightface.app import FaceAnalysis
from insightface.utils import face_align

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
# Indica ao notebook to render figures in-page.
%matplotlib inline

## 1. A Biblioteca InsightFace

O `FaceAnalysis` carrega um "pacote" de modelos ONNX, baixado automaticamente do repositório oficial no primeiro uso. O pacote padrão, `buffalo_l`, reúne:

* `det_10g` (RetinaFace): [**detecção**] caixa delimitadora e 5 pontos de referência (*keypoints*) por rosto.
* `2d106det` / `1k3d68`: pontos de referência mais densos (106 e 68 pontos, em 2D e 3D).
* `genderage`: estimativa de idade e gênero.
* `w600k_r50` (ArcFace): [**reconhecimento**] um vetor de *embedding* por rosto, usado para comparar identidades.

Este notebook cobre apenas detecção, seleção da pessoa principal e alinhamento (tarefas que dependem só da caixa delimitadora e dos 5 pontos de referência). Por isso, carregamos apenas o módulo `detection` através de `allowed_modules`, evitando baixar e carregar os outros quatro modelos à toa.

### 1.1. Carregando o modelo

O parâmetro `ctx_id=0` pede para rodar na GPU 0; sem uma GPU CUDA disponível, a `insightface` recua automaticamente para a CPU. Já `det_size` é a resolução interna usada pelo detector: quanto maior, melhor a detecção de rostos pequenos ou distantes, ao custo de mais tempo de processamento.

In [ ]:
app = FaceAnalysis(name='buffalo_l', allowed_modules=['detection'])
app.prepare(ctx_id=0, det_size=(640, 640))

## 2. Detectando Múltiplas Faces

Vamos começar com a foto de um time de futebol inteiro, um bom teste para a detecção de várias faces em uma única imagem, com rostos de tamanhos e ângulos diferentes.

### 2.1. Carregando a imagem

In [ ]:
image_path = 'imagens/05/selecao-brasileira.webp'

img = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 8))
plt.imshow(img_rgb)
plt.axis('off')
plt.show()

### 2.2. Detectando os rostos

O método `get()` devolve uma lista de objetos `Face`, um por rosto encontrado. Cada `Face` traz `bbox` (caixa delimitadora), `kps` (5 pontos de referência) e `det_score` (confiança da detecção).

In [ ]:
faces = app.get(img)

print(f"{len(faces)} rostos detectados")

for i, face in enumerate(faces):
    x1, y1, x2, y2 = face.bbox.round().astype(int)
    print(f"Rosto {i}: caixa=({x1}, {y1}, {x2}, {y2}), confiança={face.det_score:.2f}")

### 2.3. Apresentando os resultados

Criamos uma função de apresentação para desenhar a caixa delimitadora e os 5 pontos de referência de cada rosto para ser reaproveitada no restante do notebook.

In [ ]:
def desenhar_rostos(img: np.ndarray, faces: list, cor: tuple = (0, 255, 0), espessura: int = 2) -> np.ndarray:
    """ Desenha a caixa delimitadora, a confiança e os pontos de referência de cada rosto """
    img = img.copy()

    for face in faces:
        x1, y1, x2, y2 = face.bbox.round().astype(int)
        cv2.rectangle(img, (x1, y1), (x2, y2), cor, espessura)
        cv2.putText(
            img,
            f"{face.det_score:.2f}",
            (x1, max(y1 - 8, 0)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            cor,
            espessura
        )
        for x, y in face.kps.round().astype(int):
            cv2.circle(img, (x, y), 2, cor, -1)

    return img

In [ ]:
img_anotada = desenhar_rostos(img, faces)

plt.figure(figsize=(12, 10))
plt.imshow(cv2.cvtColor(img_anotada, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

## 3. Destacando a Pessoa Principal

Em fotos com várias pessoas, como selfies em grupo, muitas vezes só interessa identificar a pessoa "principal" da imagem: quem tirou a foto, ou está mais perto da câmera. A `insightface` não devolve isso diretamente, mas dá para estimar com uma heurística simples: entre os rostos detectados, a pessoa principal costuma ser aquela cujo rosto ocupa a maior área da imagem. Quanto mais perto da câmera, maior o rosto.

### 3.1. Carregando a imagem e detectando os rostos

In [ ]:
image_path = 'imagens/05/selfie-multiplas-pessoas.jpg'

img = cv2.imread(image_path)
faces = app.get(img)

print(f"{len(faces)} rostos detectados")

img_anotada = desenhar_rostos(img, faces)

plt.figure(figsize=(8, 14))
plt.imshow(cv2.cvtColor(img_anotada, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

### 3.2. Selecionando pelo maior rosto

In [ ]:
def area_rosto(face) -> float:
    """ Calcula a área da caixa delimitadora de um rosto """
    x1, y1, x2, y2 = face.bbox
    return (x2 - x1) * (y2 - y1)

pessoa_principal = max(faces, key=area_rosto)

print(f"Rosto principal: área={area_rosto(pessoa_principal):.0f}px², confiança={pessoa_principal.det_score:.2f}")

### 3.3. Apresentando o resultado

Desenhamos todos os rostos detectados em cinza, fino, e destacamos a pessoa principal com uma caixa mais grossa e colorida.

In [ ]:
img_anotada = desenhar_rostos(img, faces, cor=(160, 160, 160), espessura=1)
img_anotada = desenhar_rostos(img_anotada, [pessoa_principal], cor=(0, 140, 255), espessura=3)

plt.figure(figsize=(8, 10))
plt.imshow(cv2.cvtColor(img_anotada, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

## 4. Alinhamento de Faces

Rostos aparecem em fotos com poses, rotações e escalas diferentes: a cabeça inclinada, os olhos fechados, ângulos de perfil. Para tarefas posteriores, como comparar ou reconhecer rostos, essa variação atrapalha: o mesmo rosto, fotografado de ângulos diferentes, gera recortes muito diferentes entre si.

O **alinhamento** resolve isso: a partir dos 5 pontos de referência (`kps`) devolvidos pela detecção (olho esquerdo, olho direito, nariz, e os cantos esquerdo e direito da boca), a `insightface` calcula a transformação de similaridade (rotação, escala e translação) que melhor leva esses 5 pontos para um conjunto fixo de posições de referência, definido pelo padrão do ArcFace. O resultado é um recorte de 112x112 pixels onde olhos, nariz e boca sempre caem, aproximadamente, nas mesmas coordenadas, não importa a pose original do rosto.

### 4.1. Carregando a imagem e detectando os rostos

A imagem é uma grade 2x3 com o mesmo rosto fotografado em seis poses diferentes, ideal para comparar o efeito do alinhamento em ângulos e inclinações distintas.

In [ ]:
image_path = 'imagens/05/exemplos-selfie.jpg'

img = cv2.imread(image_path)
faces = app.get(img)

print(f"{len(faces)} rostos detectados")

img_anotada = desenhar_rostos(img, faces)

plt.figure(figsize=(8, 14))
plt.imshow(cv2.cvtColor(img_anotada, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

### 4.2. Recorte simples x recorte alinhado

Para comparar cada um dos seis rostos de forma justa, primeiro os ordenamos na ordem de leitura da grade (linha por linha, esquerda para a direita), agrupando pela linha a partir da posição vertical da caixa delimitadora. Depois, geramos dois recortes por rosto:

* **Recorte simples**: só a região da caixa delimitadora, redimensionada para 112x112, sem nenhuma correção de rotação ou escala.
* **Recorte alinhado**: obtido com `face_align.norm_crop()`, que usa os 5 pontos de referência para levar o rosto à pose padrão do ArcFace.

In [ ]:
altura_img = img.shape[0]
faces_ordenadas = sorted(faces, key=lambda face: (int(face.bbox[1] // (altura_img / 3)), face.bbox[0]))

def recorte_simples(img: np.ndarray, face, size: int = 112) -> np.ndarray:
    """ Recorta a caixa delimitadora do rosto e redimensiona, sem nenhum alinhamento """
    x1, y1, x2, y2 = face.bbox.round().astype(int)
    x1, y1 = max(x1, 0), max(y1, 0)
    recorte = img[y1:y2, x1:x2]
    return cv2.resize(recorte, (size, size))

recortes_simples = [recorte_simples(img, face) for face in faces_ordenadas]
recortes_alinhados = [face_align.norm_crop(img, landmark=face.kps, image_size=112) for face in faces_ordenadas]

### 4.3. Apresentando todos os alinhamentos

In [ ]:
fig, eixos = plt.subplots(2, len(faces_ordenadas), figsize=(2.2 * len(faces_ordenadas), 5))

for i, (simples, alinhado) in enumerate(zip(recortes_simples, recortes_alinhados)):
    eixos[0, i].imshow(cv2.cvtColor(simples, cv2.COLOR_BGR2RGB))
    eixos[0, i].set_xticks([])
    eixos[0, i].set_yticks([])

    eixos[1, i].imshow(cv2.cvtColor(alinhado, cv2.COLOR_BGR2RGB))
    eixos[1, i].set_xticks([])
    eixos[1, i].set_yticks([])

eixos[0, 0].set_ylabel("Simples")
eixos[1, 0].set_ylabel("Alinhado")

plt.tight_layout()
plt.show()

Repare como, na linha de baixo, os olhos e a boca caem sempre nas mesmas alturas aproximadas em todos os recortes, mesmo nos rostos originalmente inclinados, de perfil ou com a cabeça apoiada na mão. Na linha de cima, sem alinhamento, essa referência varia de recorte para recorte, seguindo a pose original de cada foto.

É por isso que o alinhamento é considerado um pré-processamento padrão antes de comparar ou reconhecer rostos: ele remove a variação de pose da equação, deixando as diferenças remanescentes atribuíveis de fato à identidade da pessoa.